<a href="https://colab.research.google.com/github/uranovnass/Machine-Learning/blob/main/ml9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# Генерация синтетического датасета "полумесяцы"
X2, y2 = make_moons(
    n_samples=300,
    noise=0.25,
    random_state=42
)

# Масштабирование признаков
X2 = StandardScaler().fit_transform(X2)

# Проверка размерностей данных
print(f"Объектов: {len(X2)}, классы: {np.bincount(y2)}")

Объектов: 300, классы: [150 150]


In [10]:
def plot_boundary(model, X, y, title, fname):

    # Создание сетки 300x300
    xx, yy = np.meshgrid(
        np.linspace(-3, 3, 300),
        np.linspace(-3, 3, 300)
    )

    # Предсказание класса для каждой точки сетки
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(5, 4))

    # Закрашивание областей классов
    plt.contourf(
        xx,
        yy,
        zz,
        alpha=0.25,
        cmap="coolwarm"
    )

    # Отображение объектов
    plt.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        cmap="coolwarm",
        s=18,
        edgecolors="k"
    )

    # Опорные векторы
    sv = model.support_vectors_

    plt.scatter(
        sv[:, 0],
        sv[:, 1],
        s=70,
        facecolors="none",
        edgecolors="k"
    )

    plt.title(
        f"{title}: train acc={model.score(X, y):.3f}, "
        f"SV={len(sv)}"
    )

    plt.tight_layout()
    plt.savefig(fname, dpi=120)
    plt.close()

In [11]:
# Создание и обучение SVM с RBF-ядром
rbf = SVC(
    kernel="rbf",
    C=1,
    gamma="scale"
)

rbf.fit(X2, y2)

# Построение графика
plot_boundary(
    rbf,
    X2,
    y2,
    "rbf, C=1",
    "svm_rbf.png"
)

# Метрики
print(
    "rbf: acc=%.3f опорных=%d"
    % (
        rbf.score(X2, y2),
        len(rbf.support_vectors_)
    )
)

rbf: acc=0.910 опорных=92


In [12]:
print("\n--- Подбор параметра C в цикле ---")

for C in [0.01, 1, 100]:

    # Создание модели
    m = SVC(
        kernel="rbf",
        C=C,
        gamma="scale"
    )

    # Обучение
    m.fit(X2, y2)

    # Визуализация
    plot_boundary(
        m,
        X2,
        y2,
        f"rbf, C={C}",
        f"svm_rbf_C{C}.png"
    )

    # Результаты
    print(
        f"C={C:<5} "
        f"acc={m.score(X2, y2):.3f} "
        f"опорных={len(m.support_vectors_)}"
    )


--- Подбор параметра C в цикле ---
C=0.01  acc=0.870 опорных=300
C=1     acc=0.910 опорных=92
C=100   acc=0.937 опорных=57


In [14]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC


# Загрузка данных
df = pd.read_csv("sample_data/o_churn_synthetic.csv")

print("Размер данных:", df.shape)
print(df.head())


# Признаки
X = df.drop(columns=["churn", "subscriber_code"])

# Целевая переменная
y = df["churn"]


# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# Препроцессинг
prep = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["region", "tariff"]
        )
    ],
    remainder=StandardScaler()
)


# Pipeline: preprocessing + SVM
svm = make_pipeline(
    prep,
    SVC(kernel="rbf")
)

print("Pipeline SVM успешно создан.")

Размер данных: (3000, 13)
   subscriber_code           region   tariff  tenure_months  \
0           503715           Бишкек  Базовый             47   
1           844126  Джалал-Абадская    Смарт             54   
2           885657          Чуйская  Базовый              1   
3           255447               Ош  Базовый             29   
4           651577               Ош    Смарт             60   

   monthly_charge_som  internet_gb  support_calls_3m  autopay  \
0                 264         12.6                 1        0   
1                 472         19.9                 0        1   
2                 291         12.9                 0        1   
3                 273          7.1                 0        0   
4                 402         14.0                 2        1   

   days_since_topup  topups_3m  dropped_calls_pct  balance_som  churn  
0                37          4               2.28          103      1  
1                12          7               1.40          1

In [15]:
import time

from sklearn.model_selection import GridSearchCV, StratifiedKFold


print("\n--- Запуск GridSearchCV ---")


# Сетка параметров
grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": [0.001, 0.01, 0.1, 1]
}


# Кросс-валидация
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# GridSearch
search = GridSearchCV(
    svm,
    grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)


# Запуск
t0 = time.perf_counter()

search.fit(X_train, y_train)

elapsed = time.perf_counter() - t0


# Лучшие параметры
print(
    f"Лучшее: {search.best_params_} "
    f"AUC CV={search.best_score_:.3f}"
)

print(f"Поиск: {elapsed:.0f} c")


# Таблица результатов
table = pd.DataFrame(
    search.cv_results_
).pivot(
    index="param_svc__C",
    columns="param_svc__gamma",
    values="mean_test_score"
)

print("\nСводная таблица ROC-AUC:")
print(table.round(3))


--- Запуск GridSearchCV ---
Лучшее: {'svc__C': 10, 'svc__gamma': 0.001} AUC CV=0.777
Поиск: 35 c

Сводная таблица ROC-AUC:
param_svc__gamma  0.001  0.010  0.100  1.000
param_svc__C                                
0.1               0.771  0.772  0.763  0.655
1.0               0.774  0.776  0.745  0.655
10.0              0.777  0.761  0.694  0.646
100.0             0.774  0.739  0.646  0.646


In [16]:
# ==========================================
# СЛАЙД 8: СРАВНЕНИЕ ПЯТИ МОДЕЛЕЙ
# ==========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


# Препроцессинг для моделей
tree_prep = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["region", "tariff"]
        )
    ],
    remainder="passthrough"
)


# Получаем лучшие параметры SVM напрямую
best_C = search.best_params_["svc__C"]
best_gamma = search.best_params_["svc__gamma"]

print("Лучшие параметры SVM:")
print("C =", best_C)
print("gamma =", best_gamma)


# Создаем пять моделей
models = {

    "SVM rbf (лучшая)": make_pipeline(
        prep,
        SVC(
            kernel="rbf",
            C=best_C,
            gamma=best_gamma
        )
    ),

    "SVM Linear": make_pipeline(
        prep,
        SVC(
            kernel="linear",
            C=1
        )
    ),

    "Логрегрессия": make_pipeline(
        prep,
        LogisticRegression(
            max_iter=1000
        )
    ),

    "Случайный лес": make_pipeline(
        tree_prep,
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )
    ),

    "XGBoost": make_pipeline(
        tree_prep,
        XGBClassifier(
            n_estimators=127,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
            eval_metric="logloss"
        )
    )
}


print("\nМодели успешно созданы:")

for name in models:
    print("-", name)

Лучшие параметры SVM:
C = 10
gamma = 0.001

Модели успешно созданы:
- SVM rbf (лучшая)
- SVM Linear
- Логрегрессия
- Случайный лес
- XGBoost


In [17]:
# ==========================================
# СЛАЙД 9: ТЕСТ НА МАСШТАБИРУЕМОСТЬ
# ==========================================

from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC
import time


print("\n--- Сравнение времени обучения ---")


# Проверяем разные размеры выборки
for n in [1000, 3000, 5000]:

    # Создание искусственного датасета
    Xn, yn = make_classification(
        n_samples=n,
        n_features=20,
        random_state=42
    )

    # -----------------------------
    # SVM с RBF
    # -----------------------------
    t0 = time.perf_counter()

    SVC(
        kernel="rbf"
    ).fit(Xn, yn)

    t_rbf = time.perf_counter() - t0


    # -----------------------------
    # LinearSVC
    # -----------------------------
    t0 = time.perf_counter()

    LinearSVC(
        dual="auto",
        random_state=42
    ).fit(Xn, yn)

    t_linear = time.perf_counter() - t0


    # Вывод
    print(
        f"n={n:<6} "
        f"RBF SVC: {t_rbf:.2f} c   "
        f"LinearSVC: {t_linear:.2f} c"
    )


--- Сравнение времени обучения ---
n=1000   RBF SVC: 0.05 c   LinearSVC: 0.01 c
n=3000   RBF SVC: 0.17 c   LinearSVC: 0.01 c
n=5000   RBF SVC: 0.83 c   LinearSVC: 0.01 c


In [18]:
# ==========================================
# СЛАЙД 10: ФИНАЛЬНОЕ СРАВНЕНИЕ МОДЕЛЕЙ
# ==========================================

from sklearn.metrics import f1_score, roc_auc_score


print("\n--- Финальные результаты на тестовой выборке ---")


# Перебираем все модели из Слайда 8
for name, model in models.items():

    # -----------------------------
    # Обучение
    # -----------------------------
    t0 = time.perf_counter()

    model.fit(X_train, y_train)

    elapsed = time.perf_counter() - t0


    # -----------------------------
    # Предсказание классов
    # -----------------------------
    preds = model.predict(X_test)


    # -----------------------------
    # Получаем score для ROC-AUC
    # -----------------------------
    final_estimator = model.steps[-1][1]

    if hasattr(final_estimator, "decision_function"):

        scores = model.decision_function(X_test)

    else:

        scores = model.predict_proba(X_test)[:, 1]


    # -----------------------------
    # F1
    # -----------------------------
    f1 = f1_score(
        y_test,
        preds
    )


    # -----------------------------
    # ROC-AUC
    # -----------------------------
    auc = roc_auc_score(
        y_test,
        scores
    )


    # -----------------------------
    # Вывод
    # -----------------------------
    print(
        f"{name:<20} "
        f"F1={f1:.3f}   "
        f"ROC-AUC={auc:.3f}   "
        f"Time={elapsed:.2f} c"
    )



--- Финальные результаты на тестовой выборке ---
SVM rbf (лучшая)     F1=0.599   ROC-AUC=0.801   Time=0.21 c
SVM Linear           F1=0.599   ROC-AUC=0.798   Time=0.29 c
Логрегрессия         F1=0.611   ROC-AUC=0.806   Time=0.03 c
Случайный лес        F1=0.556   ROC-AUC=0.752   Time=0.55 c
XGBoost              F1=0.586   ROC-AUC=0.792   Time=0.13 c
